# Hetzner CLI 101

An interactive walkthrough of the `hcloud` CLI — Hetzner Cloud's command-line tool for managing servers, SSH keys, images, and more.

**Convention used in this notebook:**
- Cells without a marker are **read-only / safe** — they list or describe resources, never modify anything.
- Cells marked **⚠️ LIVE** will create, modify, or delete real Hetzner Cloud resources. They include a confirmation step or are commented out by default.

**Cost note:** The resources we create (CX22 server, snapshot) cost cents per hour. The teardown section at the end cleans everything up. Don't forget to run it when you're done!

---
## 1. Setup & Prerequisites

Before we begin, let's make sure `hcloud` is installed and we have a valid API token configured.

If you haven't installed it yet:
- **macOS:** `brew install hcloud`
- **Linux:** `snap install hcloud` or grab the binary from [GitHub releases](https://github.com/hetznercloud/cli/releases)
- **Windows:** `scoop install hcloud` or download from releases

In [ ]:
# Check hcloud is installed and show version
hcloud version

### Configure your API token

You need a Hetzner Cloud API token. Generate one at [console.hetzner.cloud](https://console.hetzner.cloud/) → your project → Security → API Tokens → Generate API Token (Read & Write).

Set it as an environment variable so the notebook can use it. **Never hardcode tokens in notebooks you share.**

In [ ]:
# Option A: Set token from environment (recommended)
# Export HCLOUD_TOKEN in your shell before launching Jupyter:
#   export HCLOUD_TOKEN="your-token-here"

# Option B: Set it here for this session only (don't commit this!)
# export HCLOUD_TOKEN="your-token-here"

# Verify we have a token set (shows first/last 4 chars only)
if [ -z "$HCLOUD_TOKEN" ]; then
  echo "❌ HCLOUD_TOKEN is not set. Please set it before continuing."
else
  echo "✅ HCLOUD_TOKEN is set: ${HCLOUD_TOKEN:0:4}...${HCLOUD_TOKEN: -4}"
fi

In [ ]:
# List configured contexts — confirms the CLI can talk to Hetzner
hcloud context list

> **Tip:** If you use multiple Hetzner projects, `hcloud context create <name>` lets you switch between them. Each context stores a separate API token.

---
## 2. Orientation — What's Available?

Before creating anything, let's explore what Hetzner offers. These are all read-only calls — no resources are created or modified.

### Server Types

Server types define the CPU, RAM, and disk configuration. The naming convention: `CX` = shared vCPU (Intel/AMD), `CAX` = Arm64 (Ampere), `CCX` = dedicated vCPU.

In [ ]:
# List all available server types with pricing-relevant specs
hcloud server-type list -o columns=name,description,cores,memory,disk,storage_type

### Locations & Datacenters

Hetzner has datacenters in Germany (Falkenstein, Nuremberg), Finland (Helsinki), USA (Ashburn, Hillsboro), and Singapore. For EU-based workloads you'll typically pick `fsn1`, `nbg1`, or `hel1`.

In [ ]:
hcloud location list

In [ ]:
hcloud datacenter list

### Operating System Images

These are the base images you can use when creating a server. Hetzner maintains the system images; you can also create your own snapshots (we'll do that later).

In [ ]:
# System images (maintained by Hetzner)
hcloud image list --type system -o columns=id,name,description,os_flavor,architecture,disk_size

In [ ]:
# Your snapshots (if any)
hcloud image list --type snapshot

---
## 3. SSH Key Management

SSH keys are stored at the project level in Hetzner Cloud. When you create a server, you reference these keys by name — Hetzner injects them into the server's `authorized_keys` automatically.

In [ ]:
# List SSH keys registered in this project (safe — read only)
hcloud ssh-key list

### ⚠️ LIVE — Upload an SSH Key

This uploads your local public key to Hetzner Cloud so we can use it when creating servers. If you already have a key registered, you can skip this cell.

In [ ]:
# ⚠️ LIVE — Uncomment and run to upload your SSH key
# Adjust the path and name if needed

# DEMO_SSH_KEY_NAME="demo-notebook-key"
# hcloud ssh-key create --name "$DEMO_SSH_KEY_NAME" --public-key-from-file ~/.ssh/id_ed25519.pub

echo "ℹ️  Uncomment the lines above and run again to upload your key."

In [ ]:
# Verify key was uploaded
hcloud ssh-key list

---
## 4. Server Lifecycle — The Main Event

This is the core workflow: create a server, inspect it, connect to it, and tear it down. We'll use a cheap CX22 (2 vCPU, 4 GB RAM) in Falkenstein.

**Define some variables first** — adjust these to your preference:

In [ ]:
# Demo variables — adjust as needed
DEMO_SERVER_NAME="demo-notebook-01"
DEMO_SERVER_TYPE="cx22"
DEMO_IMAGE="ubuntu-24.04"
DEMO_LOCATION="fsn1"
DEMO_SSH_KEY_NAME="demo-notebook-key"  # Must match a key from section 3

echo "Server: $DEMO_SERVER_NAME ($DEMO_SERVER_TYPE)"
echo "Image:  $DEMO_IMAGE @ $DEMO_LOCATION"
echo "SSH:    $DEMO_SSH_KEY_NAME"

### ⚠️ LIVE — Create the Server

This provisions a real server. Billing starts immediately (CX22 ≈ €0.006/hour). We'll destroy it at the end.

In [ ]:
# ⚠️ LIVE — Creates a real server
# Uncomment the hcloud line to run

# hcloud server create \
#   --name "$DEMO_SERVER_NAME" \
#   --type "$DEMO_SERVER_TYPE" \
#   --image "$DEMO_IMAGE" \
#   --location "$DEMO_LOCATION" \
#   --ssh-key "$DEMO_SSH_KEY_NAME"

echo "ℹ️  Uncomment the hcloud command above and run again to create the server."

### Inspect the Server

Once created, we can query its details — IP address, status, specs.

In [ ]:
# Show server details
hcloud server describe "$DEMO_SERVER_NAME"

In [ ]:
# Grab the public IPv4 for SSH
DEMO_IP=$(hcloud server ip "$DEMO_SERVER_NAME")
echo "Server IP: $DEMO_IP"

### SSH Smoke Test

Let's verify we can connect. The first connection may take 10–20 seconds while the server finishes booting.

In [ ]:
# Quick SSH test — runs `hostname` on the remote server
# If this fails, wait 30 seconds and retry (server may still be booting)
ssh -o StrictHostKeyChecking=no -o ConnectTimeout=10 root@"$DEMO_IP" hostname

In [ ]:
# Check what we're running on
ssh -o StrictHostKeyChecking=no root@"$DEMO_IP" 'uname -a && cat /etc/os-release | head -4'

### Server Power Controls

You can stop, start, reboot, and reset servers from the CLI. These are safe to run (they affect only the demo server).

In [ ]:
# Check current status
hcloud server list -o columns=name,status,ipv4,server_type,datacenter

In [ ]:
# ⚠️ LIVE — Power off the server (graceful shutdown)
# hcloud server shutdown "$DEMO_SERVER_NAME"

echo "ℹ️  Uncomment to shut down the server."

In [ ]:
# ⚠️ LIVE — Power it back on
# hcloud server poweron "$DEMO_SERVER_NAME"

echo "ℹ️  Uncomment to power the server back on."

---
## 5. Snapshots & Images

Snapshots capture the full disk state of a server. Useful for backups before risky changes, creating golden images, or cloning servers. Snapshots are billed at €0.0119/GB/month.

In [ ]:
# List existing snapshots (safe)
hcloud image list --type snapshot

### ⚠️ LIVE — Create a Snapshot

This creates a snapshot of our demo server. The server should ideally be powered off for a clean snapshot, but Hetzner supports live snapshots too (with a brief filesystem freeze).

In [ ]:
# ⚠️ LIVE — Create a snapshot of the demo server
# hcloud server create-image --type snapshot --description "demo-notebook-snapshot" "$DEMO_SERVER_NAME"

echo "ℹ️  Uncomment to create a snapshot."

In [ ]:
# Verify snapshot was created
hcloud image list --type snapshot -o columns=id,description,disk_size,created

### ⚠️ LIVE — Delete the Snapshot

Clean up — we don't want orphaned snapshots accumulating charges.

In [ ]:
# ⚠️ LIVE — Delete the snapshot
# Replace <SNAPSHOT_ID> with the ID from the list above
# hcloud image delete <SNAPSHOT_ID>

echo "ℹ️  Replace <SNAPSHOT_ID> and uncomment to delete."

---
## 6. Cleanup / Teardown

**⚠️ IMPORTANT: Run this section when you're done to avoid ongoing charges.**

We'll delete the server and any remaining snapshots created during this demo.

In [ ]:
# Pre-flight check — list what we're about to delete
echo "=== Servers ==="
hcloud server list -o columns=name,status,ipv4,server_type
echo ""
echo "=== Snapshots ==="
hcloud image list --type snapshot -o columns=id,description,disk_size,created
echo ""
echo "=== SSH Keys ==="
hcloud ssh-key list

In [ ]:
# ⚠️ LIVE — Delete the demo server
# hcloud server delete "$DEMO_SERVER_NAME"

echo "ℹ️  Uncomment to delete the demo server."

In [ ]:
# ⚠️ LIVE — Remove the demo SSH key (optional — only if you uploaded one in section 3)
# hcloud ssh-key delete "$DEMO_SSH_KEY_NAME"

echo "ℹ️  Uncomment to remove the demo SSH key."

In [ ]:
# Final verification — should show nothing demo-related
echo "=== Remaining servers ==="
hcloud server list
echo ""
echo "=== Remaining snapshots ==="
hcloud image list --type snapshot
echo ""
echo "✅ Cleanup complete. No billable demo resources should remain."

---
## 7. Next Steps

Now that you've seen the basics, here are some directions to explore:

- **Volumes:** Persistent block storage that survives server deletion → `hcloud volume --help`
- **Firewalls:** Network-level rules applied to servers → `hcloud firewall --help`
- **Load Balancers:** Distribute traffic across servers → `hcloud load-balancer --help`
- **Private Networks:** Internal VLANs between servers → `hcloud network --help`
- **Terraform Provider:** Infrastructure-as-code with [hetznercloud/hcloud](https://registry.terraform.io/providers/hetznercloud/hcloud/latest)
- **API Docs:** Full REST API at [docs.hetzner.cloud](https://docs.hetzner.cloud/)

### Shell Completion

For daily use, enable tab completion:

In [ ]:
# Show completion setup instructions for your shell
hcloud completion --help

### Output Formats

The CLI supports multiple output formats — useful for scripting and piping into other tools:

In [ ]:
# Table (default)
hcloud server-type list -o columns=name,cores,memory | head -5

echo "---"

# JSON — pipe to jq for scripting
hcloud server-type list -o json | python3 -c "import sys,json; d=json.load(sys.stdin); print(f'{len(d)} server types available')"

echo "---"

# YAML
hcloud location list -o yaml | head -10

---
*Built for Yorizon — interactive `hcloud` CLI walkthrough.*